In [1]:
import redis
import os
from dotenv import load_dotenv

load_dotenv()

# Connect to Redis
# decode_responses=True means Redis returns Python strings instead of bytes
r = redis.Redis(
    host=os.getenv('REDIS_HOST', 'localhost'),
    port=int(os.getenv('REDIS_PORT', 6379)),
    decode_responses=True
)

# Test the connection with a PING
try:
    response = r.ping()
    info = r.info()
    print(f"Connected to Redis!")
    print(f"  Redis version : {info['redis_version']}")
    print(f"  Memory used   : {info['used_memory_human']}")
    print(f"  Keys stored   : {r.dbsize()}")
except redis.ConnectionError as e:
    print(f"Could not connect to Redis: {e}")
    print("  Make sure Docker Desktop is running and you ran: docker-compose up -d")

Connected to Redis!
  Redis version : 7.4.7
  Memory used   : 1.35M
  Keys stored   : 0


In [2]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

load_dotenv(override=True)

## LLM
llm = ChatOpenAI(
    model=os.getenv("EVREN_MODEL"),          # deepseek-v4-flash
    api_key=os.getenv("EVREN_API_KEY"),     
    base_url=os.getenv("EVREN_BASE_URL"),
    temperature=0    # https://evren-llmapi.ssyz.org.tr/v1
)   


## llm Test
response = llm.invoke([HumanMessage(content="Are u there?!")])
print("LLM Response:", response.content)

LLM Response: Yes, I’m here! How can I help you? 😊


In [3]:
# ─────────────────────────────────────────────
# STRINGS — The most basic Redis data type
# Use for: single values, cached responses
# ─────────────────────────────────────────────

print("\n" + "═" * 55)
print("\033[1;36m  🔤  REDIS STRINGS\033[0m")
print("\033[90m  For single values and cached responses\033[0m")
print("═" * 55)

# SET stores any string value under a key
r.set("llm:model", "deepseek-v4-flash")
r.set("llm:temperature", "0.7")

# GET retrieves a value by key
model = r.get("llm:model")
temperature = r.get("llm:temperature")

print(f"\n  \033[1;33m▸ llm:model\033[0m        → \033[1;32m{model}\033[0m")
print(f"  \033[1;33m▸ llm:temperature\033[0m  → \033[1;32m{temperature}\033[0m")

# EXISTS checks if a key is present (returns 1 or 0)
exists_model = bool(r.exists("llm:model"))
exists_fake = bool(r.exists("fake:key"))

print(f"\n  \033[1;34m🔍 Existence Check\033[0m")
print(f"     Is 'llm:model' key present?  → \033[1;32m{exists_model}\033[0m")
print(f"     Is 'fake:key'  key present?  → \033[1;31m{exists_fake}\033[0m")

print("═" * 55 + "\n")


═══════════════════════════════════════════════════════
  🔤  REDIS STRINGS
  For single values and cached responses
═══════════════════════════════════════════════════════

  ▸ llm:model        → deepseek-v4-flash
  ▸ llm:temperature  → 0.7

  🔍 Existence Check
     Is 'llm:model' key present?  → True
     Is 'fake:key'  key present?  → False
═══════════════════════════════════════════════════════



In [4]:
r.delete("llm:model","llm:temperature" )
print(f"\nAfter deletion, 'llm model' exists? {bool(r.exists('course:name'))}")


After deletion, 'llm model' exists? False


## TTL

In [5]:
import time

r.set("demo:expiring", "I will disappear!", ex=10)

print(f"Key stored! TTL = {r.ttl('demo:expiring')} seconds")
print(f"GO CHECK REDIS DASHBOARD NOW — you will see 'demo:expiring'")
print("\nWaiting 10 seconds...")
time.sleep(10)

print(f"\nValue now: {r.get('demo:expiring')}")
print("REFRESH DASHBOARD — key is gone!")
import time

r.set("demo:expiring", "I will disappear!", ex=10)

print(f"Key stored! TTL = {r.ttl('demo:expiring')} seconds")
print(f"GO CHECK REDIS DASHBOARD NOW — you will see 'demo:expiring'")
print("\nWaiting 10 seconds...")
time.sleep(10)

print(f"\nValue now: {r.get('demo:expiring')}")
print("REFRESH DASHBOARD — key is gone!")


Key stored! TTL = 10 seconds
GO CHECK REDIS DASHBOARD NOW — you will see 'demo:expiring'

Waiting 10 seconds...

Value now: None
REFRESH DASHBOARD — key is gone!
Key stored! TTL = 10 seconds
GO CHECK REDIS DASHBOARD NOW — you will see 'demo:expiring'

Waiting 10 seconds...

Value now: None
REFRESH DASHBOARD — key is gone!


In [6]:
import json

# ─────────────────────────────────────────────
# HASHES — Store structured data (like a dict)
# Use for: caching a response WITH its metadata
# ─────────────────────────────────────────────

print("=" * 45)
print("REDIS HASHES")
print("=" * 45)

# HSET stores multiple fields under one key
r.hset("llm:response:sample", mapping={
    "prompt": "What is machine learning?",
    "response": "Machine learning is a branch of AI where systems learn from data to make predictions or decisions without being explicitly programmed for each task.",
    "model": "deepseek-v4-flash",
    "tokens": "52",
    "cost_usd": "0.000008"
})
print("Stored hash with 5 fields under key: llm:response:sample")

# HGET retrieves one specific field
response_text = r.hget("llm:response:sample", "response")
print(f"\nJust the response field:")
print(f"  {response_text}")

# HGETALL retrieves all fields as a Python dict
all_fields = r.hgetall("llm:response:sample")
print(f"\nAll fields:")
for field, value in all_fields.items():
    display = value if len(value) <= 60 else value[:60] + "..."
    print(f"  {field:12} : {display}")

# r.delete("llm:response:sample")

REDIS HASHES
Stored hash with 5 fields under key: llm:response:sample

Just the response field:
  Machine learning is a branch of AI where systems learn from data to make predictions or decisions without being explicitly programmed for each task.

All fields:
  tokens       : 52
  model        : deepseek-v4-flash
  prompt       : What is machine learning?
  response     : Machine learning is a branch of AI where systems learn from ...
  cost_usd     : 0.000008


In [7]:
# ─────────────────────────────────────────────
# LISTS — Ordered sequences
# Use for: conversation history (ordered messages)
# ─────────────────────────────────────────────

print("=" * 45)
print("REDIS LISTS")
print("=" * 45)

# Each message is stored as a JSON string
conversation = [
    {"role": "system", "content": "You are a helpful AI assistant."},
    {"role": "user", "content": "Hi! What is Redis?"},
    {"role": "assistant", "content": "Redis is an in-memory data store used for caching."},
    {"role": "user", "content": "Why is it fast?"},
    {"role": "assistant", "content": "Because it stores everything in RAM, not on disk."},
]

# RPUSH adds items to the RIGHT (end) of the list — preserves order
for msg in conversation:
    r.rpush("demo:conversation", json.dumps(msg))

print(f"Pushed {len(conversation)} messages to 'demo:conversation'")
print(f"List length: {r.llen('demo:conversation')} messages")

# LRANGE retrieves items by index (0, -1 means everything)
raw_messages = r.lrange("demo:conversation", 0, -1)
parsed = [json.loads(m) for m in raw_messages]

print("\nFull conversation history:")
for msg in parsed:
    print(f"  [{msg['role'].upper():10}] {msg['content']}")

# LTRIM keeps only a slice of the list
# This is how we implement a "sliding window" memory — keep last N messages
r.ltrim("demo:conversation", -3, -1)  # Keep only the last 3 messages
trimmed = [json.loads(m) for m in r.lrange("demo:conversation", 0, -1)]
print(f"\nAfter LTRIM (keep last 3): {len(trimmed)} messages")
for msg in trimmed:
    print(f"  [{msg['role'].upper():10}] {msg['content']}")

# r.delete("demo:conversation")

REDIS LISTS
Pushed 5 messages to 'demo:conversation'
List length: 5 messages

Full conversation history:
  [SYSTEM    ] You are a helpful AI assistant.
  [USER      ] Hi! What is Redis?
  [ASSISTANT ] Redis is an in-memory data store used for caching.
  [USER      ] Why is it fast?
  [ASSISTANT ] Because it stores everything in RAM, not on disk.

After LTRIM (keep last 3): 3 messages
  [ASSISTANT ] Redis is an in-memory data store used for caching.
  [USER      ] Why is it fast?
  [ASSISTANT ] Because it stores everything in RAM, not on disk.
